## Example 1: Basic classification schemes and choropleth mapping

In [ ]:
import mapclassify
import numpy as np
import pandas as pd

def assert_valid(classifier, y, k=None):
    """Invariants every mapclassify classifier satisfies."""
    assert classifier.counts.sum() == len(y), (classifier.counts, len(y))
    assert (np.diff(classifier.bins) >= 0).all(), classifier.bins
    assert len(classifier.bins) == classifier.k, (classifier.bins, classifier.k)
    if k is not None:
        assert classifier.k == k, (classifier.k, k)
    assert classifier.yb.min() >= 0 and classifier.yb.max() < len(classifier.bins), classifier.yb

# Example 1: Basic classification schemes and choropleth mapping
print("mapclassify module imported successfully")

try:
    print("Example 1 - Basic classification schemes and choropleth mapping")
    
    # Generate sample geographic data (e.g., county population densities)
    np.random.seed(42)
    n_regions = 50
    population_density = np.random.lognormal(mean=5, sigma=1, size=n_regions)
    income_data = np.random.normal(50000, 15000, n_regions)
    
    # 1. Equal Interval Classification
    equal_interval = mapclassify.EqualInterval(population_density, k=5)
    print(f"Equal Interval Classification (5 classes):")
    print(f"Bin ranges: {equal_interval.bins}")
    print(f"Class assignments: {equal_interval.yb[:10]}...")  # First 10 assignments
    print(f"Class counts: {equal_interval.counts}")
    
    # 2. Quantiles Classification
    quantiles = mapclassify.Quantiles(population_density, k=5)
    print(f"\nQuantiles Classification (5 classes):")
    print(f"Bin ranges: {quantiles.bins}")
    print(f"Class assignments: {quantiles.yb[:10]}...")
    print(f"Class counts: {quantiles.counts}")
    
    # 3. Natural Breaks (Jenks)
    natural_breaks = mapclassify.NaturalBreaks(population_density, k=5)
    print(f"\nNatural Breaks (Jenks) Classification:")
    print(f"Bin ranges: {natural_breaks.bins}")
    print(f"Class assignments: {natural_breaks.yb[:10]}...")
    print(f"Class counts: {natural_breaks.counts}")
    print(f"Goodness of Variance Fit: {natural_breaks.adcm:.3f}")
    
    # 4. Standard Deviations Classification
    std_dev = mapclassify.StdMean(income_data, multiples=[-2, -1, 0, 1, 2])
    print(f"\nStandard Deviations Classification:")
    print(f"Mean: {income_data.mean():.0f}, Std: {income_data.std():.0f}")
    print(f"Bin ranges: {std_dev.bins}")
    print(f"Class counts: {std_dev.counts}")
    
    for classifier in (equal_interval, quantiles, natural_breaks):
        assert_valid(classifier, population_density, k=5)
    assert_valid(std_dev, income_data)
    assert quantiles.counts.tolist() == [10] * 5  # 50 distinct values split into equal-count classes
    
    print("mapclassify example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mapclassify example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced classification algorithms and optimization

In [ ]:
import mapclassify
import numpy as np
import pandas as pd

def assert_valid(classifier, y, k=None):
    """Invariants every mapclassify classifier satisfies."""
    assert classifier.counts.sum() == len(y), (classifier.counts, len(y))
    assert (np.diff(classifier.bins) >= 0).all(), classifier.bins
    assert len(classifier.bins) == classifier.k, (classifier.bins, classifier.k)
    if k is not None:
        assert classifier.k == k, (classifier.k, k)
    assert classifier.yb.min() >= 0 and classifier.yb.max() < len(classifier.bins), classifier.yb

# Example 2: Advanced classification algorithms and optimization
print("mapclassify advanced classification algorithms")

try:
    print("Example 2 - Advanced classification algorithms and optimization")
    
    # Generate complex multimodal data
    np.random.seed(123)
    
    # Simulate economic data with distinct clusters
    cluster1 = np.random.normal(25000, 3000, 30)    # Low income cluster
    cluster2 = np.random.normal(45000, 5000, 40)    # Middle income cluster  
    cluster3 = np.random.normal(75000, 8000, 20)    # High income cluster
    cluster4 = np.random.normal(120000, 15000, 10)  # Very high income cluster
    
    economic_data = np.concatenate([cluster1, cluster2, cluster3, cluster4])
    
    # 1. Fisher-Jenks Natural Breaks (optimal classification)
    fisher_jenks = mapclassify.FisherJenks(economic_data, k=4)
    print(f"Fisher-Jenks Classification (4 classes):")
    print(f"Bin ranges: {fisher_jenks.bins}")
    print(f"Class counts: {fisher_jenks.counts}")
    print(f"Array Goodness of Variance Fit: {fisher_jenks.adcm:.3f}")
    
    # 2. Head-Tail Breaks (for heavy-tailed distributions)
    headtail = mapclassify.HeadTailBreaks(economic_data)
    print(f"\nHead-Tail Breaks Classification:")
    print(f"Number of classes: {headtail.k}")
    print(f"Bin ranges: {headtail.bins}")
    print(f"Class counts: {headtail.counts}")
    print(f"Head-tail ratio test: {len(economic_data[economic_data > economic_data.mean()])/len(economic_data):.2f}")
    
    # 3. Maximum Breaks Classification
    max_breaks = mapclassify.MaximumBreaks(economic_data, k=4, mindiff=1000)
    print(f"\nMaximum Breaks Classification:")
    print(f"Bin ranges: {max_breaks.bins}")
    print(f"Class counts: {max_breaks.counts}")
    
    # 4. User-Defined Classification
    custom_breaks = [30000, 50000, 80000, 110000]
    user_defined = mapclassify.UserDefined(economic_data, bins=custom_breaks)
    print(f"\nUser-Defined Classification:")
    print(f"Custom breaks: {custom_breaks}")
    print(f"Resulting bins: {user_defined.bins}")
    print(f"Class counts: {user_defined.counts}")
    
    # 5. Compare classification quality
    print(f"\nClassification Quality Comparison:")
    print(f"Fisher-Jenks ADCM: {fisher_jenks.adcm:.3f}")
    print(f"Natural Breaks ADCM: {mapclassify.NaturalBreaks(economic_data, k=4).adcm:.3f}")
    print(f"Equal Interval ADCM: {mapclassify.EqualInterval(economic_data, k=4).adcm:.3f}")
    print(f"Quantiles ADCM: {mapclassify.Quantiles(economic_data, k=4).adcm:.3f}")
    
    for classifier in (fisher_jenks, max_breaks):
        assert_valid(classifier, economic_data, k=4)
    assert_valid(headtail, economic_data)
    assert_valid(user_defined, economic_data)
    assert user_defined.bins[:4].tolist() == custom_breaks, user_defined.bins
    
    print("mapclassify example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mapclassify example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Classifier methods and pooled classification

In [ ]:
import mapclassify
import numpy as np
import pandas as pd

def assert_valid(classifier, y, k=None):
    """Invariants every mapclassify classifier satisfies."""
    assert classifier.counts.sum() == len(y), (classifier.counts, len(y))
    assert (np.diff(classifier.bins) >= 0).all(), classifier.bins
    assert len(classifier.bins) == classifier.k, (classifier.bins, classifier.k)
    if k is not None:
        assert classifier.k == k, (classifier.k, k)
    assert classifier.yb.min() >= 0 and classifier.yb.max() < len(classifier.bins), classifier.yb

# Example 3: Classifier methods and pooled classification
print("mapclassify classifier methods and pooled classification")

try:
    print("Example 3 - Classifier methods and pooled classification")
    
    # Generate time series spatial data (e.g., unemployment rates over time)
    np.random.seed(456)
    n_regions = 25
    n_years = 5
    
    # Create unemployment data for multiple years
    unemployment_data = {}
    region_names = [f"Region_{i+1}" for i in range(n_regions)]
    
    for year in range(2020, 2020 + n_years):
        # Add temporal trend and regional variation
        base_rates = np.random.uniform(3, 12, n_regions)
        seasonal_effect = 2 * np.sin((year - 2020) * np.pi / 2)  # Cyclical unemployment
        unemployment_data[year] = base_rates + seasonal_effect + np.random.normal(0, 1, n_regions)
        unemployment_data[year] = np.maximum(unemployment_data[year], 0)  # No negative unemployment
    
    # Convert to DataFrame for easier handling
    df = pd.DataFrame(unemployment_data, index=region_names)
    
    # 1. Classify each year independently
    print("Independent Year Classifications (Quantiles):")
    yearly_classifiers = {}
    for year in df.columns:
        classifier = mapclassify.Quantiles(df[year].values, k=4)
        assert_valid(classifier, df[year].values, k=4)
        yearly_classifiers[year] = classifier
        print(f"  {year}: bins = {[f'{b:.1f}' for b in classifier.bins]}, counts = {classifier.counts}")
    
    # 2. Pooled Classification (single classifier for all years)
    all_values = df.values.flatten()
    pooled_classifier = mapclassify.Quantiles(all_values, k=4)
    assert_valid(pooled_classifier, all_values, k=4)
    print(f"\nPooled Classification (all years combined):")
    print(f"Bins: {[f'{b:.1f}' for b in pooled_classifier.bins]}")
    print(f"Total observations: {len(all_values)}")
    
    # Apply pooled classifier to each year
    print("\nPooled Classification Applied to Each Year:")
    for year in df.columns:
        year_classes = pooled_classifier(df[year].values)
        assert year_classes.min() >= 0 and year_classes.max() < 4, year_classes
        unique, counts = np.unique(year_classes, return_counts=True)
        class_counts = dict(zip(unique, counts))
        print(f"  {year}: {class_counts}")
    
    # 3. Classification comparison and stability
    print(f"\nClassification Stability Analysis:")
    
    # Calculate how often regions change classes between years
    stability_matrix = np.zeros((n_regions, n_years))
    for i, year in enumerate(df.columns):
        stability_matrix[:, i] = yearly_classifiers[year](df[year].values)
    
    # Count class changes for each region
    class_changes = 0
    for i in range(n_regions):
        changes = np.sum(np.diff(stability_matrix[i, :]) != 0)
        class_changes += changes
    
    print(f"Total class changes across all regions: {class_changes}")
    print(f"Average changes per region: {class_changes / n_regions:.2f}")
    
    # 4. Custom classifier application
    print(f"\nCustom Threshold Classification:")
    # Define unemployment rate thresholds
    unemployment_thresholds = [4.0, 7.0, 10.0]  # Low, Medium, High, Very High
    threshold_classifier = mapclassify.UserDefined(all_values, bins=unemployment_thresholds)
    assert_valid(threshold_classifier, all_values)
    assert threshold_classifier.bins[:3].tolist() == unemployment_thresholds, threshold_classifier.bins
    
    print(f"Unemployment categories:")
    print(f"  Low (≤4.0%): {threshold_classifier.counts[0]} observations")
    print(f"  Medium (4.0-7.0%): {threshold_classifier.counts[1]} observations") 
    print(f"  High (7.0-10.0%): {threshold_classifier.counts[2]} observations")
    print(f"  Very High (>10.0%): {threshold_classifier.counts[3]} observations")
    
    print("mapclassify example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mapclassify example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Statistical testing and classification evaluation

In [ ]:
import mapclassify
import numpy as np
import pandas as pd

def assert_valid(classifier, y, k=None):
    """Invariants every mapclassify classifier satisfies."""
    assert classifier.counts.sum() == len(y), (classifier.counts, len(y))
    assert (np.diff(classifier.bins) >= 0).all(), classifier.bins
    assert len(classifier.bins) == classifier.k, (classifier.bins, classifier.k)
    if k is not None:
        assert classifier.k == k, (classifier.k, k)
    assert classifier.yb.min() >= 0 and classifier.yb.max() < len(classifier.bins), classifier.yb

# Example 4: Statistical testing and classification evaluation
print("mapclassify statistical testing and evaluation")

try:
    print("Example 4 - Statistical testing and classification evaluation")
    
    # Generate realistic demographic and economic data
    np.random.seed(789)
    n_counties = 100
    
    # Create correlated demographic variables
    education_levels = np.random.beta(2, 5, n_counties) * 100  # % with college degree
    median_income = 30000 + education_levels * 800 + np.random.normal(0, 5000, n_counties)
    median_income = np.maximum(median_income, 20000)  # Minimum income floor
    
    poverty_rates = np.maximum(0, 25 - education_levels * 0.3 + np.random.normal(0, 3, n_counties))
    crime_rates = np.maximum(0, poverty_rates * 2.5 + np.random.normal(0, 5, n_counties))
    
    # Create DataFrame
    data = pd.DataFrame({
        'education': education_levels,
        'income': median_income,
        'poverty': poverty_rates,
        'crime': crime_rates,
        'county_id': [f'County_{i+1}' for i in range(n_counties)]
    })
    
    print(f"Dataset created with {n_counties} counties")
    print(f"Income range: ${data.income.min():.0f} - ${data.income.max():.0f}")
    print(f"Education range: {data.education.min():.1f}% - {data.education.max():.1f}%")
    
    # 1. Multiple classification schemes comparison
    classification_methods = {
        'Equal Interval': mapclassify.EqualInterval(data.income, k=5),
        'Quantiles': mapclassify.Quantiles(data.income, k=5),
        'Natural Breaks': mapclassify.NaturalBreaks(data.income, k=5),
        'Fisher Jenks': mapclassify.FisherJenks(data.income, k=5),
        'Std Mean': mapclassify.StdMean(data.income, multiples=[-1.5, -0.5, 0.5, 1.5])
    }
    
    for method_name, classifier in classification_methods.items():
        assert_valid(classifier, data.income, k=None if method_name == 'Std Mean' else 5)
    
    print(f"\nClassification Methods Comparison (Income Data):")
    for method_name, classifier in classification_methods.items():
        print(f"\n{method_name}:")
        print(f"  Bins: {[f'${b:,.0f}' for b in classifier.bins]}")
        print(f"  Counts: {classifier.counts}")
        print(f"  ADCM: {classifier.adcm:.3f}")
    
    # 2. Outlier detection and handling
    print(f"\nOutlier Detection and Classification:")
    
    # BoxPlot classification (identifies outliers)
    boxplot_classifier = mapclassify.BoxPlot(data.income)
    assert_valid(boxplot_classifier, data.income)
    print(f"BoxPlot Classification:")
    print(f"  Classes: {boxplot_classifier.k}")
    print(f"  Bins: {[f'${b:,.0f}' for b in boxplot_classifier.bins]}")
    print(f"  Outlier class (class 4): {boxplot_classifier.counts[4] if len(boxplot_classifier.counts) > 4 else 0} counties")
    
    # Percentile-based classification
    percentile_classifier = mapclassify.Percentiles(data.income, pct=[10, 25, 75, 90])
    assert_valid(percentile_classifier, data.income)
    print(f"\nPercentiles Classification (10th, 25th, 75th, 90th):")
    print(f"  Bins: {[f'${b:,.0f}' for b in percentile_classifier.bins]}")
    print(f"  Counts: {percentile_classifier.counts}")
    
    # 3. Multi-variable classification analysis
    print(f"\nMulti-variable Classification Analysis:")
    
    # Classify multiple variables using the same scheme
    variables = ['education', 'poverty', 'crime']
    classification_results = {}
    
    for var in variables:
        classifier = mapclassify.Quantiles(data[var], k=4)
        assert_valid(classifier, data[var], k=4)
        classification_results[var] = classifier
        print(f"\n{var.title()} (Quantiles):")
        if var == 'education':
            print(f"  Bins: {[f'{b:.1f}%' for b in classifier.bins]}")
        elif var == 'poverty':
            print(f"  Bins: {[f'{b:.1f}%' for b in classifier.bins]}")
        else:  # crime
            print(f"  Bins: {[f'{b:.1f}' for b in classifier.bins]}")
        print(f"  Counts: {classifier.counts}")
    
    # 4. Classification consistency and correlation
    print(f"\nClassification Consistency Analysis:")
    
    # Create classification matrix
    class_matrix = np.zeros((n_counties, len(variables)))
    for i, var in enumerate(variables):
        class_matrix[:, i] = classification_results[var].yb
    
    # Calculate classification correlations
    from scipy.stats import spearmanr
    
    correlations = {}
    for i in range(len(variables)):
        for j in range(i+1, len(variables)):
            var1, var2 = variables[i], variables[j]
            corr, p_value = spearmanr(class_matrix[:, i], class_matrix[:, j])
            correlations[f"{var1}-{var2}"] = (corr, p_value)
            print(f"  {var1} vs {var2}: correlation = {corr:.3f}, p-value = {p_value:.3f}")
    
    # 5. Classification performance metrics
    print(f"\nClassification Performance Summary:")
    best_method = min(classification_methods.items(), key=lambda x: x[1].adcm)
    worst_method = max(classification_methods.items(), key=lambda x: x[1].adcm)
    
    print(f"  Best performing method: {best_method[0]} (ADCM = {best_method[1].adcm:.3f})")
    print(f"  Worst performing method: {worst_method[0]} (ADCM = {worst_method[1].adcm:.3f})")
    print(f"  Performance improvement: {((worst_method[1].adcm - best_method[1].adcm) / worst_method[1].adcm * 100):.1f}%")
    
    print("mapclassify example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mapclassify example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Production workflow and advanced applications

In [ ]:
import mapclassify
import numpy as np
import pandas as pd

def assert_valid(classifier, y, k=None):
    """Invariants every mapclassify classifier satisfies."""
    assert classifier.counts.sum() == len(y), (classifier.counts, len(y))
    assert (np.diff(classifier.bins) >= 0).all(), classifier.bins
    assert len(classifier.bins) == classifier.k, (classifier.bins, classifier.k)
    if k is not None:
        assert classifier.k == k, (classifier.k, k)
    assert classifier.yb.min() >= 0 and classifier.yb.max() < len(classifier.bins), classifier.yb

# Example 5: Production workflow and advanced applications
print("mapclassify production workflow and advanced applications")

try:
    print("Example 5 - Production workflow and advanced applications")
    
    # Simulate real-world GIS workflow with multiple datasets
    np.random.seed(999)
    
    # 1. Multi-scale geographic data processing
    print("Multi-scale Geographic Data Processing:")
    
    # State-level data (50 states)
    state_data = {
        'population': np.random.lognormal(15, 0.8, 50) * 1000,  # Population in thousands
        'gdp_per_capita': np.random.lognormal(10.8, 0.3, 50),   # GDP per capita
        'unemployment': np.random.beta(2, 8, 50) * 15,           # Unemployment rate %
        'area_sq_miles': np.random.lognormal(10, 1.2, 50)       # Area in sq miles
    }
    
    # County-level data (500 counties)
    county_data = {
        'median_age': np.random.normal(38, 8, 500),
        'poverty_rate': np.random.beta(1.5, 4, 500) * 30,
        'housing_cost': np.random.lognormal(12, 0.6, 500),
        'education_index': np.random.beta(3, 2, 500) * 100
    }
    
    # 2. Hierarchical classification system
    print(f"\nHierarchical Classification System:")
    
    # State-level classifications
    state_classifiers = {}
    for var, data in state_data.items():
        if var == 'unemployment':
            # Use policy-relevant thresholds for unemployment
            classifier = mapclassify.UserDefined(data, bins=[4.0, 6.0, 8.0])
            categories = ['Low', 'Moderate', 'High', 'Very High']
        else:
            classifier = mapclassify.NaturalBreaks(data, k=4)
            categories = ['Class 1', 'Class 2', 'Class 3', 'Class 4']
        
        assert_valid(classifier, data)
        state_classifiers[var] = classifier
        print(f"  State {var}: {len(categories)} classes, ADCM = {classifier.adcm:.3f}")
    
    # County-level classifications
    county_classifiers = {}
    for var, data in county_data.items():
        classifier = mapclassify.FisherJenks(data, k=5)
        assert_valid(classifier, data, k=5)
        county_classifiers[var] = classifier
        print(f"  County {var}: 5 classes, ADCM = {classifier.adcm:.3f}")
    
    # 3. Classification validation and robustness testing
    print(f"\nClassification Robustness Testing:")
    
    # Test with different sample sizes
    test_data = county_data['poverty_rate']
    sample_sizes = [50, 100, 200, 300, 500]
    adcm_stability = []
    
    for size in sample_sizes:
        sample = np.random.choice(test_data, size, replace=False)
        classifier = mapclassify.FisherJenks(sample, k=4)
        assert_valid(classifier, sample, k=4)
        adcm_stability.append(classifier.adcm)
        print(f"  Sample size {size}: ADCM = {classifier.adcm:.4f}")
    
    # Calculate stability metric
    adcm_cv = np.std(adcm_stability) / np.mean(adcm_stability)
    print(f"  ADCM Coefficient of Variation: {adcm_cv:.4f}")
    
    # 4. Automated classification selection
    print(f"\nAutomated Classification Method Selection:")
    
    test_variable = county_data['median_age']
    methods_to_test = [
        ('Equal Interval', lambda x, k: mapclassify.EqualInterval(x, k=k)),
        ('Quantiles', lambda x, k: mapclassify.Quantiles(x, k=k)),
        ('Natural Breaks', lambda x, k: mapclassify.NaturalBreaks(x, k=k)),
        ('Fisher Jenks', lambda x, k: mapclassify.FisherJenks(x, k=k)),
        ('Maximum Breaks', lambda x, k: mapclassify.MaximumBreaks(x, k=k))
    ]
    
    k_values = [3, 4, 5, 6]
    best_combinations = []
    
    for k in k_values:
        method_results = []
        for method_name, method_func in methods_to_test:
            classifier = method_func(test_variable, k)
            assert_valid(classifier, test_variable, k=k)
            method_results.append((method_name, k, classifier.adcm))
        
        if method_results:
            best_method = min(method_results, key=lambda x: x[2])
            best_combinations.append(best_method)
            print(f"  k={k}: Best method = {best_method[0]} (ADCM = {best_method[2]:.4f})")
    
    # Overall best configuration
    if best_combinations:
        overall_best = min(best_combinations, key=lambda x: x[2])
        print(f"  Overall best: {overall_best[0]} with k={overall_best[1]} (ADCM = {overall_best[2]:.4f})")
    
    # 5. Performance and scalability analysis
    print(f"\nPerformance and Scalability Analysis:")
    
    import time
    
    # Test performance with different data sizes
    performance_results = {}
    data_sizes = [100, 500, 1000, 2000]
    
    for size in data_sizes:
        large_dataset = np.random.lognormal(10, 1, size)
        
        # Time different classification methods
        method_times = {}
        for method_name, method_func in methods_to_test[:4]:  # Skip MaximumBreaks for large datasets
            start_time = time.time()
            classifier = method_func(large_dataset, 5)
            end_time = time.time()
            assert_valid(classifier, large_dataset, k=5)
            method_times[method_name] = (end_time - start_time) * 1000  # Convert to milliseconds
        
        performance_results[size] = method_times
        fastest_method = min(method_times.items(), key=lambda x: x[1])
        print(f"  Size {size}: Fastest = {fastest_method[0]} ({fastest_method[1]:.2f}ms)")
    
    # 6. Export classification results for GIS
    print(f"\nGIS Integration Workflow:")
    
    # Create final classification for export
    final_classifier = mapclassify.FisherJenks(county_data['poverty_rate'], k=4)
    assert_valid(final_classifier, county_data['poverty_rate'], k=4)
    
    # Simulate creating classification labels and colors
    class_labels = ['Low Poverty', 'Moderate Poverty', 'High Poverty', 'Very High Poverty']
    class_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Colorbrewer-style colors
    
    print(f"  Classification scheme: {len(class_labels)} classes")
    print(f"  Class breaks: {[f'{b:.1f}%' for b in final_classifier.bins]}")
    print(f"  Class counts: {final_classifier.counts}")
    print(f"  Ready for GIS export with color scheme")
    
    # Summary statistics
    total_classifiers_created = len(state_classifiers) + len(county_classifiers) + len(best_combinations) + 1
    print(f"\nWorkflow Summary:")
    print(f"  Total classifiers created: {total_classifiers_created}")
    print(f"  Classification methods tested: {len(methods_to_test)}")
    print(f"  Performance tests completed: {len(data_sizes)} size variations")
    print(f"  Robustness tests: {len(sample_sizes)} sample sizes")
    
    print("mapclassify example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mapclassify example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")
print(f"\nMapclassify capabilities demonstrated:")
print("- Basic choropleth classification schemes (quantiles, equal intervals, natural breaks)")
print("- Advanced algorithms (Fisher-Jenks, Head-Tail Breaks, user-defined)")
print("- Pooled classification and temporal analysis")
print("- Statistical evaluation and multi-variable analysis")  
print("- Production workflows with performance optimization and GIS integration")